# Shop&Shop 일일 업무 기록 v2
### 사용법
1. **셀 1 실행** → 구글 드라이브 마운트 (세션 시작할 때마다)
2. **셀 2 실행** → 순서대로 입력 → 드라이브 CSV 누적 저장

### 저장 위치
- `내 드라이브 > shopnshop > chicken_stock_data.csv`

### 주의
- 세션 끊겨도 구글 드라이브에 저장되므로 데이터 유실 없음
- 같은 날짜 중복 입력 시 덮어쓰기 여부 확인
- 16일치 모이면 CSV 다운로드 → Workbench import

In [ ]:
# ── 구글 드라이브 마운트 ──────────────────────────
# 세션 시작할 때마다 이 셀 먼저 실행
# 최초 실행 시 권한 허용 팝업 → 허용 클릭

from google.colab import drive
import os

drive.mount('/content/drive')

FOLDER_PATH = '/content/drive/MyDrive/shopnshop'
os.makedirs(FOLDER_PATH, exist_ok=True)

CSV_PATH = f'{FOLDER_PATH}/chicken_stock_data.csv'
print(f'✅ 드라이브 마운트 완료')
print(f'📁 저장 경로: {CSV_PATH}')


Mounted at /content/drive
✅ 드라이브 마운트 완료
📁 저장 경로: /content/drive/MyDrive/shopnshop/chicken_stock_data.csv


In [ ]:
import pandas as pd
import os

# ── 입력 ──────────────────────────────────────────
print('=' * 40)
print('   Shop&Shop 일일 업무 기록')
print('=' * 40)

날짜 = pd.to_datetime(input('날짜 (yyyy-mm-dd): ')).strftime('%Y-%m-%d')
요일 = ['월','화','수','목','금','토','일'][pd.to_datetime(날짜).weekday()]
dcl_id = 'dcl_' + pd.to_datetime(날짜).strftime('%m%d')

날씨_비여부 = int(input('비 여부 (0=맑음, 1=비): '))
공휴일여부  = int(input('공휴일 여부 (0=N, 1=Y): '))

if 공휴일여부 == 1:
    연휴여부 = int(input('연휴 여부 (0=N, 1=Y): '))
    if 연휴여부 == 1:
        연휴_상황 = int(input('연휴 위치 (0=첫날, 1=중간, 2=끝날): '))
    else:
        연휴_상황 = '-'
    연휴전날여부 = '-'
else:
    연휴여부  = '-'
    연휴_상황 = '-'
    if 요일 == '금':
        휴일전날여부 = 1
        print('휴일 전날 여부 여부: 1 (금요일 자동 적용)')
    else:
        휴일전날여부 = int(input('휴일 전날 여부 (0=N, 1=Y): '))

특수일 = int(input('특수일 (0=없음, 1=복날, 2=스포츠이벤트): '))

대타여부    = int(input('대타 여부 (0=N, 1=Y): '))

초벌_생닭kg = float(input('1차 초벌량 생닭 (kg): '))
초벌_완성kg = round(초벌_생닭kg * 0.88, 2)

재초벌여부  = int(input('재초벌 여부 (0=N, 1=Y): '))

if 재초벌여부 == 1:
    재초벌전잔량_g = int(input('재초벌 직전 잔량 (g): '))
    재초벌시각     = input('재초벌 시각 (HH:MM): ')
    재초벌_생닭kg  = float(input('재초벌량 생닭 (kg): '))
    재초벌_완성kg  = round(재초벌_생닭kg * 0.88, 2)
else:
    재초벌전잔량_g = '-'
    재초벌시각     = '-'
    재초벌_생닭kg  = '-'
    재초벌_완성kg  = '-'

마감잔량_g  = int(input('마감 잔량 (g): '))
퇴근시각    = input('퇴근 시각 (HH:MM): ')
특이사항    = input('특이사항 (없으면 엔터): ').strip() or '-'
완벌조리_kg = float(input('완벌조리(kg) 없으면 0  :'))

# ── 행 생성 ───────────────────────────────────────
new_row = pd.DataFrame([{
    'dcl_id':         dcl_id,
    '날짜':             날짜,
    '요일':             요일,
    '날씨_비여부':      날씨_비여부,
    '공휴일여부':       공휴일여부,
    '연휴여부':         연휴여부,
    '연휴_상황':        연휴_상황,
    '휴일전날여부':     휴일전날여부,
    '특수일':           특수일,
    '대타여부':         대타여부,
    '1차초벌량_생닭kg': 초벌_생닭kg,
    '1차초벌량_완성kg': 초벌_완성kg,
    '재초벌여부':       재초벌여부,
    '재초벌전잔량_g':   재초벌전잔량_g,
    '재초벌시각':       재초벌시각,
    '재초벌량_생닭kg':  재초벌_생닭kg,
    '재초벌량_완성kg':  재초벌_완성kg,
    '마감잔량_g':       마감잔량_g,
    '퇴근시각':         퇴근시각,
    '특이사항':         특이사항,
    '완벌조리kg': 완벌조리_kg
}])

# ── CSV 누적 저장 ─────────────────────────────────
if os.path.exists(CSV_PATH):
    existing = pd.read_csv(CSV_PATH)
    if 날짜 in existing['날짜'].values:
        print(f'\n⚠ {날짜} 데이터가 이미 존재합니다. 덮어쓰시겠어요? (y/n): ', end='')
        if input().strip().lower() == 'y':
            existing = existing[existing['날짜'] != 날짜]
        else:
            print('저장 취소.')
            raise SystemExit
    combined = pd.concat([existing, new_row], ignore_index=True)
else:
    combined = new_row

combined.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')

print(f'\n✅ 저장 완료! ({날짜} / dcl_id: {dcl_id} / 누적 {len(combined)}일치)')
print(new_row.T.to_string())

   Shop&Shop 일일 업무 기록
날짜 (yyyy-mm-dd): 2026-07-13
비 여부 (0=맑음, 1=비): 0
공휴일 여부 (0=N, 1=Y): 0
휴일 전날 여부 (0=N, 1=Y): 0
특수일 (0=없음, 1=복날, 2=스포츠이벤트): 0
대타 여부 (0=N, 1=Y): 0
1차 초벌량 생닭 (kg): 20
재초벌 여부 (0=N, 1=Y): 0
마감 잔량 (g): 367
퇴근 시각 (HH:MM): 23:00
특이사항 (없으면 엔터): 
완벌조리(kg) 없으면 0  :0

✅ 저장 완료! (2026-07-13 / dcl_id: dcl_0713 / 누적 29일치)
                     0
dcl_id        dcl_0713
날짜          2026-07-13
요일                   월
날씨_비여부               0
공휴일여부                0
연휴여부                 -
연휴_상황                -
휴일전날여부               0
특수일                  0
대타여부                 0
1차초벌량_생닭kg        20.0
1차초벌량_완성kg        17.6
재초벌여부                0
재초벌전잔량_g             -
재초벌시각                -
재초벌량_생닭kg            -
재초벌량_완성kg            -
마감잔량_g             367
퇴근시각             23:00
특이사항                 -
완벌조리kg             0.0
